# Text-Only Cheating Detection: Full Experimental Sweep

Goal: find the **best text-only model** for cheating detection. No WavLM, no audio embeddings. Just transcript-derived features.

What this notebook runs:
1. Whisper transcription (cached)
2. ~50 features in 7 groups (text, pause, suspicious, formal/AI, prosodic, voice quality, perplexity)
3. **Ablation**: train on each feature group alone + incremental
4. **Model comparison**: XGBoost vs LightGBM vs RandomForest vs LogisticRegression
5. **Top-N feature selection**: curve of F1 vs number of features
6. **Final winner**: threshold sweep + precision-first points
7. Head-to-head vs your WavLM audio results

Install (optional deps are optional -- features degrade to 0):
```
pip install faster-whisper xgboost lightgbm spacy librosa
python -m spacy download en_core_web_sm
pip install praat-parselmouth transformers torch  # optional
```

In [ ]:
# ================================================================
# CONFIGURATION
# ================================================================
from pathlib import Path

TRAIN_FOLDERS = ["audios2", "audios4"]
TEST_FOLDER   = "audios5"

WHISPER_MODEL  = "small"
FILLER_PROMPT  = ("Umm, let me think like, hmm... Okay here's what I'm thinking. "
                  "So uh, basically, you know, I mean, like, right.")

# Feature group toggles (set False to skip heavy steps during fast iteration)
USE_PROSODIC       = True    # f0, energy from librosa (slow-ish)
USE_VOICE_QUALITY  = True    # jitter/shimmer/HNR (needs parselmouth)
USE_PERPLEXITY     = True    # GPT-2 perplexity (needs transformers)

TEST_RATIO  = 0.20
RANDOM_SEED = 42
AUDIO_EXTS  = {".wav",".mp3",".m4a",".flac",".ogg",".wma",".aac",".webm",".mp4"}

NB_DIR   = Path(".").resolve()
SAVE_DIR = NB_DIR / "checkpoints_text"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# WavLM baseline (from wavlm_4way_comparison, audios2+4 -> audios5)
WAVLM_BASELINE = {
    "Whole+Pre":  dict(precision=0.7368, recall=0.5385, f1=0.6222),
    "Whole+FT":   dict(precision=0.5833, recall=0.6731, f1=0.6250),
    "Seg+Pre":    dict(precision=0.5806, recall=0.6923, f1=0.6316),
    "Seg+FT":     dict(precision=0.6538, recall=0.6538, f1=0.6538),
}

LABEL_MAP = {
    "read":1,"cheating":1,"reading":1,"scripted":1,"yes":1,"1":1,1:1,
    "spontaneous":0,"not cheating":0,"not_cheating":0,"no":0,"0":0,0:0,"genuine":0,
}
print(f"Train: {TRAIN_FOLDERS}  |  Test: {TEST_FOLDER or f'{int(TEST_RATIO*100)}% split'}")
print(f"Whisper: {WHISPER_MODEL}  |  Prosodic: {USE_PROSODIC}  VoiceQ: {USE_VOICE_QUALITY}  Perp: {USE_PERPLEXITY}")

In [ ]:
import os, re, json, warnings
import numpy as np
import pandas as pd
from collections import Counter
from tqdm import tqdm
warnings.filterwarnings('ignore')
np.random.seed(RANDOM_SEED)

import xgboost as xgb
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (precision_score, recall_score, f1_score,
                              accuracy_score, confusion_matrix, classification_report)

try:
    import lightgbm as lgb
    HAS_LGBM = True
except ImportError:
    HAS_LGBM = False
    print('LightGBM not installed -- will skip.')

try:
    import spacy
    nlp = spacy.load('en_core_web_sm', disable=['ner','lemmatizer'])
    HAS_SPACY = True
except (ImportError, OSError):
    HAS_SPACY = False
    print('spaCy not available -- POS features will be skipped.')

import librosa

if USE_VOICE_QUALITY:
    try:
        import parselmouth
        from parselmouth.praat import call as praat_call
        HAS_PARSELMOUTH = True
    except ImportError:
        HAS_PARSELMOUTH = False
        print('parselmouth not installed -- voice quality features will be 0.')
else:
    HAS_PARSELMOUTH = False

if USE_PERPLEXITY:
    try:
        import torch
        from transformers import GPT2LMHeadModel, GPT2TokenizerFast
        _gpt2_tok = None; _gpt2_mdl = None
        def _gpt2():
            global _gpt2_tok, _gpt2_mdl
            if _gpt2_mdl is None:
                _gpt2_tok = GPT2TokenizerFast.from_pretrained('gpt2')
                _gpt2_mdl = GPT2LMHeadModel.from_pretrained('gpt2').eval()
            return _gpt2_mdl, _gpt2_tok
        HAS_GPT2 = True
    except ImportError:
        HAS_GPT2 = False
        print('transformers not available -- perplexity features will be 0.')
else:
    HAS_GPT2 = False

print(f'Deps: lgbm={HAS_LGBM} spacy={HAS_SPACY} praat={HAS_PARSELMOUTH} gpt2={HAS_GPT2}')

## 1. Feature Definitions (7 Groups)

Each feature belongs to exactly one group. This powers the ablation study.

In [ ]:
FILLERS           = {'um','uh','uh-huh','uhm','umm','hmm','hm','er','ah','ehm','mhm'}
DISCOURSE_MARKERS = {'you know','i mean','like','basically','actually','so','well','right','okay','oh','anyway','honestly'}
HEDGES            = {'i think','i guess','maybe','perhaps','probably','kind of','sort of','i believe','it seems','i suppose','might be'}
SELF_REF          = {'i','me','my','myself','mine',"i'm","i've","i'd","i'll"}
REPAIRS           = ['i mean','no wait','sorry i','actually no','wait no','no no']
FORMAL_TRANS      = ['furthermore','moreover','however','therefore','additionally','consequently',
                     'nevertheless','hence','thus','in conclusion','firstly','secondly','thirdly',
                     'in summary','to summarize','in essence','overall','ultimately']
AI_PHRASES        = ['it is important to note','it is worth noting','it should be noted',
                     'in conclusion','to summarize','in summary','fundamentally',
                     'plays a crucial role','plays a vital role','a wide range of',
                     'on the other hand','in other words','delve into','it is crucial']
CONTENT_POS       = {'NOUN','VERB','ADJ','ADV','PROPN'}
FUNCTION_POS      = {'DET','ADP','CONJ','CCONJ','SCONJ','PRON','AUX','PART'}

# Feature group definitions
G_DISFLUENCY  = ['filler_rate','filler_count','repetition_rate','repair_rate',
                 'discourse_marker_rate','hedge_rate']
G_STYLOMETRIC = ['ttr','mattr','mtld','complex_word_rate','avg_word_length',
                 'n_words','n_unique_words','avg_sentence_length','std_sentence_length',
                 'fragment_rate','n_sentences','self_ref_rate',
                 'noun_rate','verb_rate','adj_rate']
G_PAUSE       = ['pause_mean','pause_std','pause_median','pause_skew','long_pause_rate',
                 'pause_ratio','n_pauses','pause_regularity',
                 'pause_before_content_ratio','pause_before_function_ratio',
                 'mid_phrase_pause_rate','words_per_sec','articulation_rate',
                 'initial_pause','longest_pause']
G_SUSPICIOUS  = ['suspicious_gap_count','suspicious_gap_ratio']
G_FORMAL_AI   = ['formal_transition_count','formal_transition_rate',
                 'ai_phrase_count','ai_phrase_rate']
G_PROSODIC    = ['f0_mean','f0_std','f0_range','f0_skew','f0_slope',
                 'energy_mean','energy_std','speaking_rate_std']
G_VOICE_Q     = ['jitter_local','shimmer_local','hnr_mean']
G_PERPLEXITY  = ['mean_perplexity','burstiness']

GROUPS = {
    'disfluency':  G_DISFLUENCY,
    'stylometric': G_STYLOMETRIC,
    'pause':       G_PAUSE,
    'suspicious':  G_SUSPICIOUS,
    'formal_ai':   G_FORMAL_AI,
    'prosodic':    G_PROSODIC,
    'voice_q':     G_VOICE_Q,
    'perplexity':  G_PERPLEXITY,
}
ALL_FEATURES = [f for g in GROUPS.values() for f in g]
print(f'{len(ALL_FEATURES)} features across {len(GROUPS)} groups')
for g, feats in GROUPS.items():
    print(f'  {g:12s}: {len(feats):2d} features')

In [ ]:
# ================================================================
# FEATURE FUNCTIONS
# ================================================================
WORD_RE = re.compile(r"[a-zA-Z']+")

def _syllable_count(word):
    word = word.lower().strip()
    if len(word) <= 3: return 1
    count, prev_vowel = 0, False
    for ch in word:
        iv = ch in 'aeiouy'
        if iv and not prev_vowel: count += 1
        prev_vowel = iv
    if word.endswith('e') and count > 1: count -= 1
    return max(count, 1)

def _mattr(words, window=50):
    if len(words) < window: return len(set(words)) / max(len(words), 1)
    return float(np.mean([len(set(words[i:i+window])) / window for i in range(len(words)-window+1)]))

def _mtld(words, threshold=0.72):
    if len(words) < 10: return 0.0
    def _run(ws):
        factors, current = 0, []
        for w in ws:
            current.append(w)
            ttr = len(set(current)) / len(current)
            if ttr <= threshold:
                factors += 1; current = []
        if current:
            ttr = len(set(current)) / len(current)
            if ttr < 1.0:
                factors += (1.0 - ttr) / (1.0 - threshold)
        return len(ws) / factors if factors > 0 else len(ws)
    return round((_run(words) + _run(words[::-1])) / 2, 2)

def compute_text_and_stylo(text):
    out = {k: 0 for k in G_DISFLUENCY + G_STYLOMETRIC}
    if not text or len(text.strip()) < 10: return out
    text_lower = text.lower().strip()
    if HAS_SPACY:
        doc = nlp(text_lower)
        words = [t.text for t in doc if t.is_alpha]
        all_toks = [t.text for t in doc]
        sentences = list(doc.sents)
        pos_c = Counter(t.pos_ for t in doc)
    else:
        words = WORD_RE.findall(text_lower)
        all_toks = words
        sentences = [s for s in re.split(r'[.!?]+', text_lower) if s.strip()]
        pos_c = Counter()
    n_words = len(words)
    if n_words < 5: return out

    filler_count = sum(1 for w in all_toks if w in FILLERS)
    bigrams = [f'{words[i]} {words[i+1]}' for i in range(len(words)-1)]
    bc = Counter(bigrams)
    rep_rate = sum(c-1 for c in bc.values() if c>1) / max(len(bigrams),1)
    repair_c = sum(text_lower.count(r) for r in REPAIRS)
    n_sents = max(len(sentences), 1)
    sl = [len([t for t in s if getattr(t,'is_alpha',True)]) for s in sentences] if HAS_SPACY else [len(WORD_RE.findall(s)) for s in sentences]
    sl = [x for x in sl if x > 0]
    tp = sum(pos_c.values()) or 1

    out.update({
        # disfluency
        'filler_rate':       filler_count/n_words,
        'filler_count':      filler_count,
        'repetition_rate':   rep_rate,
        'repair_rate':       repair_c/n_sents,
        'discourse_marker_rate': sum(text_lower.count(d) for d in DISCOURSE_MARKERS)/n_sents,
        'hedge_rate':        sum(text_lower.count(h) for h in HEDGES)/n_sents,
        # stylometric
        'ttr':               len(set(words))/n_words,
        'mattr':             _mattr(words),
        'mtld':              _mtld(words),
        'complex_word_rate': sum(1 for w in words if _syllable_count(w)>=3)/n_words,
        'avg_word_length':   float(np.mean([len(w) for w in words])),
        'n_words':           n_words,
        'n_unique_words':    len(set(words)),
        'avg_sentence_length': float(np.mean(sl)) if sl else 0,
        'std_sentence_length': float(np.std(sl))  if len(sl)>1 else 0,
        'fragment_rate':     sum(1 for x in sl if x<4)/n_sents,
        'n_sentences':       n_sents,
        'self_ref_rate':     sum(1 for w in all_toks if w in SELF_REF)/n_words,
        'noun_rate':         pos_c.get('NOUN',0)/tp,
        'verb_rate':         pos_c.get('VERB',0)/tp,
        'adj_rate':          pos_c.get('ADJ',0)/tp,
    })
    return out

def compute_pause_and_suspicious(words):
    out = {k: 0 for k in G_PAUSE + G_SUSPICIOUS}
    if not words or len(words) < 5: return out
    pauses = []
    for i in range(1, len(words)):
        gap = words[i]['start'] - words[i-1]['end']
        if gap > 0.05:
            pauses.append({'dur': gap, 'after_word': words[i-1].get('word',''),
                           'before_word': words[i].get('word',''), 'pos': i})
    initial_pause = words[0]['start']
    all_gaps = [words[i]['start']-words[i-1]['end'] for i in range(1,len(words)) if words[i]['start']-words[i-1]['end']>0.05]
    longest_pause = max(all_gaps) if all_gaps else 0.0
    out['initial_pause'] = initial_pause
    out['longest_pause'] = longest_pause
    if not pauses: return out

    durs = [p['dur'] for p in pauses]
    total_dur = max(words[-1]['end'] - words[0]['start'], 0.1)
    speaking_dur = max(total_dur - sum(durs), 0.1)

    if HAS_SPACY:
        doc = nlp(' '.join(w.get('word','') for w in words))
        tok_pos = {t.text.lower(): t.pos_ for t in doc}
    else:
        tok_pos = {}
    nbc = nbf = nmp = 0
    for p in pauses:
        pos = tok_pos.get(p['before_word'].lower().strip('.,!?'), 'X')
        if pos in CONTENT_POS:  nbc += 1
        elif pos in FUNCTION_POS: nbf += 1
        if not p['after_word'].endswith(('.',',','!','?')): nmp += 1
    n_p = len(pauses)
    positions  = [p['pos'] for p in pauses]
    intervals  = [positions[i]-positions[i-1] for i in range(1, len(positions))]

    # Suspicious gaps: 0.3-0.8s mid-sentence (likely suppressed fillers)
    suspicious = sum(
        1 for p in pauses
        if 0.3 <= p['dur'] <= 0.8 and not p['after_word'].rstrip().endswith(('.','!','?'))
    )
    out.update({
        'pause_mean':      float(np.mean(durs)),
        'pause_std':       float(np.std(durs)),
        'pause_median':    float(np.median(durs)),
        'pause_skew':      float(pd.Series(durs).skew()) if len(durs)>2 else 0,
        'long_pause_rate': sum(1 for d in durs if d>0.5)/n_p,
        'pause_ratio':     sum(durs)/total_dur,
        'n_pauses':        n_p,
        'pause_regularity': float(np.std(intervals)) if intervals else 0,
        'pause_before_content_ratio':  nbc/n_p,
        'pause_before_function_ratio': nbf/n_p,
        'mid_phrase_pause_rate':       nmp/n_p,
        'words_per_sec':   len(words)/total_dur,
        'articulation_rate': len(words)/speaking_dur,
        'initial_pause':   initial_pause,
        'longest_pause':   longest_pause,
        'suspicious_gap_count': suspicious,
        'suspicious_gap_ratio': suspicious/max(len(words),1),
    })
    return out

def compute_formal_ai(text):
    out = {k: 0 for k in G_FORMAL_AI}
    if not text: return out
    tl = text.lower()
    n_words = max(len(WORD_RE.findall(tl)), 1)
    formal_c = sum(tl.count(p) for p in FORMAL_TRANS)
    ai_c     = sum(tl.count(p) for p in AI_PHRASES)
    out['formal_transition_count'] = formal_c
    out['formal_transition_rate']  = 100.0 * formal_c / n_words
    out['ai_phrase_count']         = ai_c
    out['ai_phrase_rate']          = 100.0 * ai_c / n_words
    return out

def compute_prosodic(audio_path):
    out = {k: 0 for k in G_PROSODIC}
    if not USE_PROSODIC: return out
    try:
        audio, sr = librosa.load(str(audio_path), sr=16000, mono=True, duration=120)
    except Exception: return out
    if len(audio) < 16000: return out
    f0, _, _ = librosa.pyin(audio, fmin=75, fmax=500, sr=16000, frame_length=2048)
    fv = f0[~np.isnan(f0)] if f0 is not None else np.array([])
    if len(fv) >= 10:
        slope = float(np.polyfit(np.arange(len(fv)), fv, 1)[0])
        out['f0_mean']  = float(np.mean(fv))
        out['f0_std']   = float(np.std(fv))
        out['f0_range'] = float(fv.max()-fv.min())
        out['f0_skew']  = float(pd.Series(fv).skew())
        out['f0_slope'] = slope
    rms = librosa.feature.rms(y=audio, frame_length=512, hop_length=256)[0]
    out['energy_mean'] = float(np.mean(rms))
    out['energy_std']  = float(np.std(rms))
    win = 2*16000
    rates = [float((librosa.feature.rms(y=audio[s:s+win], frame_length=512, hop_length=256)[0]
                    > np.percentile(rms,20)).mean())
             for s in range(0, len(audio)-win, 16000)]
    out['speaking_rate_std'] = float(np.std(rates)) if rates else 0
    return out

def compute_voice_quality(audio_path):
    out = {k: 0 for k in G_VOICE_Q}
    if not HAS_PARSELMOUTH: return out
    try:
        snd  = parselmouth.Sound(str(audio_path))
        pp   = praat_call(snd, 'To PointProcess (periodic, cc)', 75, 500)
        jit  = praat_call(pp,  'Get jitter (local)', 0, 0, 0.0001, 0.02, 1.3)
        shim = praat_call([snd, pp], 'Get shimmer (local)', 0, 0, 0.0001, 0.02, 1.3, 1.6)
        harm = praat_call(snd, 'To Harmonicity (cc)', 0.01, 75, 0.1, 1.0)
        hnr  = praat_call(harm, 'Get mean', 0, 0)
        out.update({'jitter_local': float(jit), 'shimmer_local': float(shim), 'hnr_mean': float(hnr)})
    except Exception: pass
    return out

def compute_perplexity(text):
    out = {k: 0 for k in G_PERPLEXITY}
    if not HAS_GPT2 or not text or len(text.strip()) < 20: return out
    try:
        mdl, tok = _gpt2()
        sents = [s for s in re.split(r'(?<=[.!?])\s+', text.strip()) if len(s.split())>3]
        if not sents: return out
        ppls = []
        for s in sents[:20]:
            enc = tok(s, return_tensors='pt', truncation=True, max_length=256)
            with torch.no_grad():
                loss = mdl(**enc, labels=enc['input_ids']).loss
            ppls.append(float(torch.exp(loss)))
        out['mean_perplexity'] = float(np.mean(ppls))
        out['burstiness']      = float(np.var(ppls))
    except Exception: pass
    return out

def compute_all_features(fp, text, words):
    r = {}
    r.update(compute_text_and_stylo(text))
    r.update(compute_pause_and_suspicious(words))
    r.update(compute_formal_ai(text))
    r.update(compute_prosodic(fp))
    r.update(compute_voice_quality(fp))
    r.update(compute_perplexity(text))
    return r

print('Feature functions loaded.')

## 2. Scan Folders

In [ ]:
def scan_folder(name):
    audio_dir = NB_DIR / name
    if not audio_dir.exists(): return None
    files = sorted(f for f in audio_dir.rglob('*') if f.suffix.lower() in AUDIO_EXTS and f.is_file())
    if not files: return None
    gt_path = NB_DIR / f'{name}GT.csv'
    if not gt_path.exists(): return None
    return {
        'name':  name,
        'files': files,
        'gt':    gt_path,
        't_json':   NB_DIR / f'{name}_transcripts.json',
        'feat_csv': NB_DIR / f'{name}_features.csv',
    }

all_names = list(dict.fromkeys(TRAIN_FOLDERS + ([TEST_FOLDER] if TEST_FOLDER else [])))
folders   = [m for m in (scan_folder(n) for n in all_names) if m]
for m in folders:
    t = 'ok' if m['t_json'].exists() else 'needed'
    f = 'ok' if m['feat_csv'].exists() else 'needed'
    print(f"  {m['name']:12s}  {len(m['files']):4d} files  transcripts={t}  features={f}")

## 3. Transcribe (cached, resume-safe)

In [ ]:
needs_whisper = any(not m['t_json'].exists() or
                    len(json.load(open(m['t_json'], encoding='utf-8'))) < len(m['files'])
                    for m in folders)

if needs_whisper:
    from faster_whisper import WhisperModel
    import torch as _torch
    device = 'cuda' if _torch.cuda.is_available() else 'cpu'
    compute_type = 'float16' if device == 'cuda' else 'int8'
    print(f'Loading Whisper {WHISPER_MODEL} ({device}, {compute_type}) ...')
    whisper = WhisperModel(WHISPER_MODEL, device=device, compute_type=compute_type)
else:
    whisper = None
    print('All transcripts cached.')

def transcribe_folder(meta):
    existing = json.load(open(meta['t_json'], encoding='utf-8')) if meta['t_json'].exists() else {}
    todo = [f for f in meta['files'] if f.name not in existing]
    if not todo:
        print(f"  {meta['name']}: {len(existing)} transcripts cached."); return
    print(f"  {meta['name']}: transcribing {len(todo)} new files ...")
    for fp in tqdm(todo, desc=meta['name']):
        try:
            segs, info = whisper.transcribe(
                str(fp), language='en', word_timestamps=True,
                initial_prompt=FILLER_PROMPT,
                vad_filter=True, vad_parameters={'min_silence_duration_ms': 100},
            )
            words, text_parts = [], []
            for seg in segs:
                text_parts.append(seg.text)
                if seg.words:
                    for w in seg.words:
                        words.append({'word': w.word.strip(), 'start': round(w.start,3), 'end': round(w.end,3)})
            existing[fp.name] = {'text': ' '.join(text_parts).strip(), 'words': words,
                                 'duration_sec': round(info.duration, 2)}
        except Exception as e:
            print(f'  FAIL {fp.name}: {e}')
            existing[fp.name] = {'text': '', 'words': [], 'duration_sec': 0}
        # save after each file
        with open(meta['t_json'], 'w', encoding='utf-8') as fh:
            json.dump(existing, fh, ensure_ascii=False, indent=1)

for m in folders: transcribe_folder(m)

## 4. Extract Features (cached, resume-safe)

Auto-patches missing columns in existing CSVs (so adding a new feature doesn't force full recompute).

In [ ]:
def extract_features_folder(meta):
    t = json.load(open(meta['t_json'], encoding='utf-8'))
    expected = set(ALL_FEATURES)
    # find files keyed in transcripts
    def row_for(fp):
        tr = t.get(fp.name, {'text':'','words':[],'duration_sec':0})
        r = compute_all_features(fp, tr.get('text',''), tr.get('words',[]))
        r['filename'] = fp.name
        r['duration_sec'] = tr.get('duration_sec', 0)
        return r

    if meta['feat_csv'].exists():
        df = pd.read_csv(meta['feat_csv'])
        have_files = set(df['filename'].tolist())
        missing_cols = expected - set(df.columns)
        if missing_cols:
            print(f"  {meta['name']}: patching {len(missing_cols)} missing columns ...")
            patches = []
            for _, r in tqdm(df.iterrows(), total=len(df), desc=f"{meta['name']} (patch)"):
                fp = next((f for f in meta['files'] if f.name == r['filename']), None)
                full = row_for(fp) if fp else {c: 0 for c in missing_cols}
                patches.append({c: full.get(c, 0) for c in missing_cols})
            df = pd.concat([df.reset_index(drop=True), pd.DataFrame(patches)], axis=1)
            df.to_csv(meta['feat_csv'], index=False)
        todo = [fp for fp in meta['files'] if fp.name not in have_files]
    else:
        todo = meta['files']

    if not todo:
        print(f"  {meta['name']}: all features extracted."); return
    print(f"  {meta['name']}: extracting {len(todo)} files ...")
    rows = []
    for fp in tqdm(todo, desc=meta['name']):
        rows.append(row_for(fp))
        if len(rows) >= 10:
            chunk = pd.DataFrame(rows)
            chunk.to_csv(meta['feat_csv'], mode='a', header=not meta['feat_csv'].exists(), index=False)
            rows = []
    if rows:
        pd.DataFrame(rows).to_csv(meta['feat_csv'], mode='a',
                                    header=not meta['feat_csv'].exists(), index=False)

for m in folders: extract_features_folder(m)

## 5. Load GT + Build Train/Test DataFrames

In [ ]:
def load_gt(gt_path):
    gt = pd.read_csv(gt_path)
    fn_col  = next(c for c in gt.columns if c.lower() in ('filename','file','name'))
    lbl_col = next(c for c in gt.columns if c.lower() in ('label','class','cheating','gt','label_int','ground_truth'))
    gt = gt.rename(columns={fn_col: 'filename', lbl_col: 'label_raw'})
    gt['label_int'] = gt['label_raw'].map(lambda x: LABEL_MAP.get(x, LABEL_MAP.get(str(x).lower().strip(), -1)))
    return gt[gt['label_int'].isin([0,1])][['filename','label_int']]

def build_df(names):
    dfs = []
    for n in names:
        meta = next((m for m in folders if m['name'] == n), None)
        if meta is None: continue
        feat = pd.read_csv(meta['feat_csv'])
        gt   = load_gt(meta['gt'])
        merged = feat.merge(gt, on='filename', how='inner')
        merged['batch'] = n
        dfs.append(merged)
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

train_df = build_df(TRAIN_FOLDERS)
test_df  = build_df([TEST_FOLDER]) if TEST_FOLDER else pd.DataFrame()

feat_cols = [c for c in ALL_FEATURES if c in train_df.columns]
if not test_df.empty:
    X_tr = train_df[feat_cols].fillna(0).values
    X_te = test_df[feat_cols].fillna(0).values
    y_tr = train_df['label_int'].values
    y_te = test_df['label_int'].values
    filenames_te = test_df['filename'].values
else:
    y_all = train_df['label_int'].values
    idx   = np.arange(len(y_all))
    tr_idx, te_idx = train_test_split(idx, test_size=TEST_RATIO, random_state=RANDOM_SEED, stratify=y_all)
    X_tr = train_df.iloc[tr_idx][feat_cols].fillna(0).values
    X_te = train_df.iloc[te_idx][feat_cols].fillna(0).values
    y_tr = y_all[tr_idx]; y_te = y_all[te_idx]
    filenames_te = train_df.iloc[te_idx]['filename'].values

spw = (y_tr==0).sum() / max((y_tr==1).sum(), 1)
print(f'Train: {len(X_tr)}  (cheating={int((y_tr==1).sum())}, honest={int((y_tr==0).sum())})')
print(f'Test:  {len(X_te)}  (cheating={int((y_te==1).sum())}, honest={int((y_te==0).sum())})')
print(f'Features used: {len(feat_cols)}   scale_pos_weight = {spw:.2f}')

## 5b. audios4-CV setup (generalisation gap)

Same protocol as `fusion_text_wavlm.ipynb`:
- `audios2` is **always in-train** (its 140 cheating examples are never discarded).
- 5-fold stratified CV runs on `audios4` rows only — `audios2` is concatenated to the train side of every fold.
- Threshold is picked **globally on the concatenated OOF vector** (not averaged across folds). F1 is non-decomposable, so per-fold best-F1 thresholds can't be averaged cleanly.
- That frozen `cv_thr` is applied once to audios5 → `gap_f1 = cv_f1 − test_f1_at_cv`.
- Class weights use `SPW_DEPLOY = (1 − 0.17)/0.17 ≈ 4.88` so the loss behaves as if deployment were 17% cheat regardless of the training fold's actual positive rate.

New columns added to every comparison table below (existing leaky columns are kept alongside):
- `cv_thr`, `cv_f1`, `cv_prec`, `cv_rec` — from the global OOF threshold
- `test_f1_at_cv`, `test_prec_at_cv`, `test_rec_at_cv` — audios5 at the frozen CV threshold
- `gap_f1` — `cv_f1 − test_f1_at_cv`
- `fold_f1_mean`, `fold_f1_std` — per-fold F1 at the global threshold (stability check)
- `fold_thr_mean`, `fold_thr_std` — per-fold best thresholds (threshold stability)

Per-fold raw numbers saved to `*_per_fold.csv` in `checkpoints_text/`.

In [ ]:
from sklearn.model_selection import StratifiedKFold

CV_TARGET_BATCH = 'audios4'
CV_FOLDS        = 5
DEPLOY_POS_RATE = 0.17
SPW_DEPLOY      = (1.0 - DEPLOY_POS_RATE) / DEPLOY_POS_RATE

if 'batch' not in train_df.columns:
    raise RuntimeError('train_df needs a "batch" column (set in build_df)')

_X_full_tr = train_df[feat_cols].fillna(0).values
_y_full_tr = train_df['label_int'].values
_mask_cv   = (train_df['batch'] == CV_TARGET_BATCH).values
_mask_a    = ~_mask_cv

X_always_full = _X_full_tr[_mask_a]
y_always      = _y_full_tr[_mask_a]
X_cv_full     = _X_full_tr[_mask_cv]
y_cv          = _y_full_tr[_mask_cv]

print('audios4-CV split:')
print(f'  always-in-train ({", ".join(b for b in TRAIN_FOLDERS if b != CV_TARGET_BATCH)}): n={len(X_always_full)}  cheat={int((y_always==1).sum())}  honest={int((y_always==0).sum())}')
print(f'  CV target ({CV_TARGET_BATCH}): n={len(X_cv_full)}  cheat={int((y_cv==1).sum())}  honest={int((y_cv==0).sum())}')
print(f'  DEPLOY_POS_RATE={DEPLOY_POS_RATE}  SPW_DEPLOY={SPW_DEPLOY:.2f}')

def _best_f1(proba, y, grid=np.arange(0.20, 0.81, 0.02)):
    bt, bf = 0.5, 0.0
    for t in grid:
        f = f1_score(y, (proba >= t).astype(int), zero_division=0)
        if f > bf: bf, bt = f, t
    return float(bt), float(bf)

def audios4_cv_oof_for_cols(col_names, build_clf, folds=CV_FOLDS, seed=RANDOM_SEED):
    idxs = [feat_cols.index(c) for c in col_names if c in feat_cols]
    if not idxs: return None, None
    X_a = X_always_full[:, idxs]; X_c = X_cv_full[:, idxs]
    oof = np.full(len(X_c), np.nan)
    fa  = np.full(len(X_c), -1, dtype=int)
    skf = StratifiedKFold(n_splits=folds, shuffle=True, random_state=seed)
    for fi, (tr_idx, va_idx) in enumerate(skf.split(X_c, y_cv)):
        if len(X_a):
            Xtr = np.vstack([X_a, X_c[tr_idx]])
            ytr = np.concatenate([y_always, y_cv[tr_idx]])
        else:
            Xtr, ytr = X_c[tr_idx], y_cv[tr_idx]
        sc  = StandardScaler().fit(Xtr)
        clf = build_clf()
        clf.fit(sc.transform(Xtr), ytr)
        oof[va_idx] = clf.predict_proba(sc.transform(X_c[va_idx]))[:, 1]
        fa[va_idx]  = fi
    return oof, fa

def cv_metrics_from_oof(oof, y, fold_assign):
    thr_g, f1_g = _best_f1(oof, y)
    pred = (oof >= thr_g).astype(int)
    global_m = dict(
        cv_thr=round(thr_g, 3),
        cv_f1=round(f1_g, 4),
        cv_prec=round(precision_score(y, pred, zero_division=0), 4),
        cv_rec=round(recall_score(y, pred, zero_division=0), 4),
    )
    per_fold = []
    for fi in sorted(set(fold_assign.tolist())):
        mk = fold_assign == fi
        y_f, p_f = y[mk], oof[mk]
        pred_f = (p_f >= thr_g).astype(int)
        own_thr, own_f1 = _best_f1(p_f, y_f)
        per_fold.append(dict(
            fold=int(fi), n=int(mk.sum()), n_cheat=int((y_f==1).sum()),
            f1_at_global=round(f1_score(y_f, pred_f, zero_division=0), 4),
            prec_at_global=round(precision_score(y_f, pred_f, zero_division=0), 4),
            rec_at_global=round(recall_score(y_f, pred_f, zero_division=0), 4),
            own_best_thr=round(own_thr, 3),
            own_best_f1=round(own_f1, 4),
        ))
    pf = pd.DataFrame(per_fold)
    agg = dict(
        fold_f1_mean=round(float(pf['f1_at_global'].mean()), 4),
        fold_f1_std =round(float(pf['f1_at_global'].std()),  4),
        fold_thr_mean=round(float(pf['own_best_thr'].mean()), 3),
        fold_thr_std =round(float(pf['own_best_thr'].std()),  3),
    )
    return global_m, agg, pf

def _test_metrics_at(proba_te, y_te_arr, thr):
    pred = (proba_te >= thr).astype(int)
    return dict(
        test_f1_at_cv  =round(f1_score(y_te_arr, pred, zero_division=0), 4),
        test_prec_at_cv=round(precision_score(y_te_arr, pred, zero_division=0), 4),
        test_rec_at_cv =round(recall_score(y_te_arr, pred, zero_division=0), 4),
    )

def cv_and_gap_bundle(col_names, build_clf, config_tag):
    """Run audios4-CV + fit on full audios2+audios4 + score audios5 at frozen cv_thr."""
    oof, fa = audios4_cv_oof_for_cols(col_names, build_clf)
    if oof is None: return {}, pd.DataFrame(), None
    g, a, pf = cv_metrics_from_oof(oof, y_cv, fa)
    idxs = [feat_cols.index(c) for c in col_names if c in feat_cols]
    X_tr_full = np.vstack([X_always_full[:, idxs], X_cv_full[:, idxs]])
    y_tr_full = np.concatenate([y_always, y_cv])
    X_te_sub  = X_te[:, idxs]
    sc  = StandardScaler().fit(X_tr_full)
    clf = build_clf()
    clf.fit(sc.transform(X_tr_full), y_tr_full)
    proba_te_cv = clf.predict_proba(sc.transform(X_te_sub))[:, 1]
    t = _test_metrics_at(proba_te_cv, y_te, g['cv_thr'])
    extra = {**g, **a, **t, 'gap_f1': round(g['cv_f1'] - t['test_f1_at_cv'], 4)}
    pf.insert(0, 'config', config_tag)
    return extra, pf, proba_te_cv

def _cv_xgb_factory():
    return xgb.XGBClassifier(
        n_estimators=400, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
        scale_pos_weight=float(SPW_DEPLOY), eval_metric='logloss',
        random_state=RANDOM_SEED)

print('audios4-CV helpers ready.')

## 6. Group-level Sanity: Which groups have strongest class separation?

For each group, compute mean absolute z-score between classes. Higher = more discriminative.

In [ ]:
sep_rows = []
for gname, feats in GROUPS.items():
    feats = [f for f in feats if f in feat_cols]
    if not feats: continue
    sub = train_df[feats + ['label_int']].fillna(0)
    mu0 = sub[sub['label_int']==0][feats].mean()
    mu1 = sub[sub['label_int']==1][feats].mean()
    sd  = sub[feats].std().replace(0, 1)
    z   = ((mu1 - mu0).abs() / sd)
    sep_rows.append({'group': gname, 'n_features': len(feats),
                     'mean_abs_z': z.mean(), 'max_abs_z': z.max(),
                     'top_feature': z.idxmax()})
sep_df = pd.DataFrame(sep_rows).sort_values('mean_abs_z', ascending=False)
print('Group discriminative power (higher = stronger class separation):')
print(sep_df.to_string(index=False))

## 7. Experiment 1 -- Ablation by Feature Group

Train XGBoost on:
- Each group alone
- All groups except one (leave-one-out)
- All features (baseline)

Metric: best-F1 threshold + precision at that point.

In [ ]:
def train_xgb(cols, X_tr, y_tr, X_te, y_te):
    idxs = [feat_cols.index(c) for c in cols if c in feat_cols]
    if not idxs: return None
    Xtr = X_tr[:, idxs]; Xte = X_te[:, idxs]
    sc = StandardScaler().fit(Xtr)
    spw_l = (y_tr==0).sum() / max((y_tr==1).sum(),1)
    m = xgb.XGBClassifier(
        n_estimators=400, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
        scale_pos_weight=spw_l, eval_metric='logloss',
        early_stopping_rounds=30, random_state=RANDOM_SEED,
    )
    m.fit(sc.transform(Xtr), y_tr, eval_set=[(sc.transform(Xte), y_te)], verbose=False)
    proba = m.predict_proba(sc.transform(Xte))[:, 1]
    best_thr, best_f1 = 0.5, 0.0
    for thr in np.arange(0.20, 0.81, 0.02):
        f = f1_score(y_te, (proba >= thr).astype(int), zero_division=0)
        if f > best_f1: best_f1, best_thr = f, thr
    pred = (proba >= best_thr).astype(int)
    cm = confusion_matrix(y_te, pred, labels=[0,1])
    return dict(
        n_feat=len(idxs), thr=best_thr,
        precision=precision_score(y_te, pred, zero_division=0),
        recall=recall_score(y_te, pred, zero_division=0),
        f1=best_f1,
        tp=int(cm[1,1]), fp=int(cm[0,1]), fn=int(cm[1,0]),
        proba=proba, model=m, scaler=sc, cols=cols,
    )

ablation       = []
ablation_folds = []

print('Training single-group models (leaky test + audios4-CV) ...')
for gname, feats in GROUPS.items():
    feats_in = [f for f in feats if f in feat_cols]
    if not feats_in: continue
    r = train_xgb(feats_in, X_tr, y_tr, X_te, y_te)
    if r is None: continue
    extra, pf, _ = cv_and_gap_bundle(feats_in, _cv_xgb_factory, f'only:{gname}')
    ablation.append({'config': f'only:{gname}',
                     **{k: r[k] for k in ('n_feat','precision','recall','f1','thr','tp','fp','fn')},
                     **extra})
    if not pf.empty: ablation_folds.append(pf)

print('Training leave-one-out models ...')
for gname, feats in GROUPS.items():
    other = [f for g, fs in GROUPS.items() if g != gname for f in fs if f in feat_cols]
    if not other: continue
    r = train_xgb(other, X_tr, y_tr, X_te, y_te)
    if r is None: continue
    extra, pf, _ = cv_and_gap_bundle(other, _cv_xgb_factory, f'drop:{gname}')
    ablation.append({'config': f'drop:{gname}',
                     **{k: r[k] for k in ('n_feat','precision','recall','f1','thr','tp','fp','fn')},
                     **extra})
    if not pf.empty: ablation_folds.append(pf)

print('Training full-feature baseline ...')
full = train_xgb(feat_cols, X_tr, y_tr, X_te, y_te)
extra_full, pf_full, _ = cv_and_gap_bundle(feat_cols, _cv_xgb_factory, 'ALL')
ablation.append({'config': 'ALL',
                 **{k: full[k] for k in ('n_feat','precision','recall','f1','thr','tp','fp','fn')},
                 **extra_full})
if not pf_full.empty: ablation_folds.append(pf_full)

abl_df       = pd.DataFrame(ablation).sort_values('f1', ascending=False)
abl_folds_df = pd.concat(ablation_folds, ignore_index=True) if ablation_folds else pd.DataFrame()

print()
print('Ablation results (leaky test F1 + audios4-CV F1 + gap), sorted by leaky F1:')
_show = ['config','n_feat','f1','cv_f1','test_f1_at_cv','gap_f1',
         'fold_f1_mean','fold_f1_std','fold_thr_mean','fold_thr_std',
         'cv_prec','cv_rec','precision','recall','thr','tp','fp','fn']
print(abl_df[[c for c in _show if c in abl_df.columns]].to_string(index=False, float_format=lambda v: f'{v:.4f}'))

## 8. Experiment 2 -- Model Comparison (all features)

In [ ]:
def eval_model(clf, Xtr, ytr, Xte, yte, name):
    sc = StandardScaler().fit(Xtr)
    clf.fit(sc.transform(Xtr), ytr)
    if hasattr(clf, 'predict_proba'):
        proba = clf.predict_proba(sc.transform(Xte))[:, 1]
    else:
        raw = clf.decision_function(sc.transform(Xte))
        proba = 1/(1+np.exp(-raw))
    best_thr, best_f1 = 0.5, 0.0
    for thr in np.arange(0.20, 0.81, 0.02):
        f = f1_score(yte, (proba >= thr).astype(int), zero_division=0)
        if f > best_f1: best_f1, best_thr = f, thr
    pred = (proba >= best_thr).astype(int)
    cm = confusion_matrix(yte, pred, labels=[0,1])
    return dict(
        model=name, thr=round(best_thr,2),
        precision=round(precision_score(yte, pred, zero_division=0),4),
        recall=round(recall_score(yte, pred, zero_division=0),4),
        f1=round(best_f1,4),
        tp=int(cm[1,1]), fp=int(cm[0,1]), fn=int(cm[1,0]),
        proba=proba, clf=clf, scaler=sc,
    )

candidates = [
    ('XGBoost', xgb.XGBClassifier(
        n_estimators=400, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
        scale_pos_weight=spw, eval_metric='logloss', random_state=RANDOM_SEED)),
    ('RandomForest', RandomForestClassifier(
        n_estimators=500, max_depth=8, min_samples_leaf=3,
        class_weight='balanced', n_jobs=-1, random_state=RANDOM_SEED)),
    ('ExtraTrees', ExtraTreesClassifier(
        n_estimators=500, max_depth=None, min_samples_leaf=2,
        class_weight='balanced', n_jobs=-1, random_state=RANDOM_SEED)),
    ('GradientBoost', GradientBoostingClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, random_state=RANDOM_SEED)),
    ('LogReg', LogisticRegression(
        C=1.0, class_weight='balanced', max_iter=2000, random_state=RANDOM_SEED)),
]
if HAS_LGBM:
    candidates.append(('LightGBM', lgb.LGBMClassifier(
        n_estimators=400, max_depth=-1, num_leaves=31, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, min_child_samples=5,
        class_weight='balanced', random_state=RANDOM_SEED, verbose=-1)))

# CV factories use SPW_DEPLOY so CV metrics match the fusion notebook's convention.
model_cv_factories = {
    'XGBoost':     lambda: xgb.XGBClassifier(
        n_estimators=400, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
        scale_pos_weight=float(SPW_DEPLOY), eval_metric='logloss', random_state=RANDOM_SEED),
    'RandomForest': lambda: RandomForestClassifier(
        n_estimators=500, max_depth=8, min_samples_leaf=3,
        class_weight={0:1.0, 1:float(SPW_DEPLOY)}, n_jobs=-1, random_state=RANDOM_SEED),
    'ExtraTrees':   lambda: ExtraTreesClassifier(
        n_estimators=500, max_depth=None, min_samples_leaf=2,
        class_weight={0:1.0, 1:float(SPW_DEPLOY)}, n_jobs=-1, random_state=RANDOM_SEED),
    'GradientBoost': lambda: GradientBoostingClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, random_state=RANDOM_SEED),
    'LogReg':       lambda: LogisticRegression(
        C=1.0, class_weight={0:1.0, 1:float(SPW_DEPLOY)}, max_iter=2000, random_state=RANDOM_SEED),
}
if HAS_LGBM:
    model_cv_factories['LightGBM'] = lambda: lgb.LGBMClassifier(
        n_estimators=400, max_depth=-1, num_leaves=31, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, min_child_samples=5,
        class_weight={0:1.0, 1:float(SPW_DEPLOY)}, random_state=RANDOM_SEED, verbose=-1)

model_results = []
model_folds   = []
for name, clf in candidates:
    r = eval_model(clf, X_tr, y_tr, X_te, y_te, name)
    factory = model_cv_factories.get(name)
    if factory is not None:
        extra, pf, _ = cv_and_gap_bundle(feat_cols, factory, f'model:{name}')
        for k, v in extra.items(): r[k] = v
        if not pf.empty: model_folds.append(pf)
    model_results.append(r)

mdl_keep = ['model','thr','precision','recall','f1','tp','fp','fn',
            'cv_thr','cv_f1','cv_prec','cv_rec',
            'test_f1_at_cv','test_prec_at_cv','test_rec_at_cv','gap_f1',
            'fold_f1_mean','fold_f1_std','fold_thr_mean','fold_thr_std']
mdl_df = pd.DataFrame([{k: r.get(k) for k in mdl_keep} for r in model_results])
mdl_df = mdl_df.sort_values('f1', ascending=False)
mdl_folds_df = pd.concat(model_folds, ignore_index=True) if model_folds else pd.DataFrame()

print('Model comparison (leaky F1 + audios4-CV F1 + gap), sorted by leaky F1:')
print(mdl_df.to_string(index=False))

## 9. Experiment 3 -- Top-N Feature Selection Curve

Use XGBoost feature importance on the full model. Train with top-N features for N = 5, 10, 15, 20, 30, all. Find the knee.

In [ ]:
imp = pd.Series(full['model'].feature_importances_, index=feat_cols).sort_values(ascending=False)
print('Top 20 features by XGBoost importance:')
print(imp.head(20).round(4).to_string())
print()

topN_rows  = []
topN_folds = []
for N in [3, 5, 10, 15, 20, 30, len(feat_cols)]:
    N = min(N, len(feat_cols))
    cols = imp.head(N).index.tolist()
    r = train_xgb(cols, X_tr, y_tr, X_te, y_te)
    extra, pf, _ = cv_and_gap_bundle(cols, _cv_xgb_factory, f'topN:{N}')
    row = {
        'N': N,
        'precision': round(r['precision'],4), 'recall': round(r['recall'],4),
        'f1': round(r['f1'],4), 'thr': round(r['thr'],2),
        'tp': r['tp'], 'fp': r['fp'], 'fn': r['fn'],
    }
    row.update(extra)
    topN_rows.append(row)
    if not pf.empty: topN_folds.append(pf)

topN_df       = pd.DataFrame(topN_rows)
topN_folds_df = pd.concat(topN_folds, ignore_index=True) if topN_folds else pd.DataFrame()

print('Top-N curve (leaky F1 + audios4-CV F1 + gap):')
_show_tn = ['N','f1','cv_f1','test_f1_at_cv','gap_f1',
            'fold_f1_mean','fold_f1_std','fold_thr_mean','fold_thr_std',
            'cv_prec','cv_rec','precision','recall','thr','tp','fp','fn']
print(topN_df[[c for c in _show_tn if c in topN_df.columns]].to_string(index=False, float_format=lambda v: f'{v:.4f}'))

## 10. Pick Winner + Precision-First Threshold Analysis

Winner = highest-F1 model from the full-feature comparison. Now sweep thresholds for precision-first deployment.

In [ ]:
winner = max(model_results, key=lambda r: r['f1'])
print(f"Winner: {winner['model']}  (best-F1 thr={winner['thr']:.2f}: Prec={winner['precision']:.4f}  Rec={winner['recall']:.4f}  F1={winner['f1']:.4f})")
print()

THRESHOLDS = np.arange(0.20, 0.81, 0.05)

def sweep(proba):
    rows = []
    for thr in THRESHOLDS:
        pred = (proba >= thr).astype(int)
        cm = confusion_matrix(y_te, pred, labels=[0,1])
        rows.append({
            'thr':  round(float(thr), 2),
            'prec': round(precision_score(y_te, pred, zero_division=0), 4),
            'rec':  round(recall_score(y_te, pred, zero_division=0), 4),
            'f1':   round(f1_score(y_te, pred, zero_division=0), 4),
            'tp':   int(cm[1,1]), 'fp': int(cm[0,1]),
            'fn':   int(cm[1,0]), 'tn': int(cm[0,0]),
        })
    return pd.DataFrame(rows)

# Full sweep for every model
sweeps = {}
for r in model_results:
    sweeps[r['model']] = sweep(r['proba'])

# Winner: full threshold table
print('='*70)
print(f'  WINNER ({winner["model"]}) -- full threshold sweep')
print('='*70)
print(sweeps[winner['model']].to_string(index=False))

# All models -- precision side-by-side
print('\n' + '='*70)
print('  PRECISION at each threshold (all models)')
print('='*70)
combined_p = pd.DataFrame({'thr': sweeps[winner['model']]['thr']})
for name, sw in sweeps.items():
    combined_p[name] = sw['prec']
print(combined_p.to_string(index=False))

# All models -- recall side-by-side
print('\n' + '='*70)
print('  RECALL at each threshold (all models)')
print('='*70)
combined_r = pd.DataFrame({'thr': sweeps[winner['model']]['thr']})
for name, sw in sweeps.items():
    combined_r[name] = sw['rec']
print(combined_r.to_string(index=False))

# Precision-first per model
print('\n' + '='*70)
print('  Precision-first operating points (>= target prec, max recall)')
print('='*70)
for target in (0.70, 0.75, 0.80, 0.85, 0.90, 0.95):
    print(f'\n  Target precision >= {target:.2f}:')
    for name, sw in sweeps.items():
        ok = sw[(sw['prec'] >= target) & (sw['tp'] >= 3)]
        if len(ok):
            best = ok.loc[ok['rec'].idxmax()]
            print(f'    {name:<16} thr={best["thr"]:.2f}  prec={best["prec"]:.4f}  rec={best["rec"]:.4f}  tp={int(best["tp"])}  fp={int(best["fp"])}')
        else:
            print(f'    {name:<16} not reached')

## 11. Head-to-head vs WavLM

In [ ]:
print(f'Train: {" + ".join(TRAIN_FOLDERS)}   Test: {TEST_FOLDER or "20% holdout"}')
print('='*62)
print(f'  {"Model":<26} {"Prec":>6} {"Rec":>6} {"F1":>6}')
print('-'*62)
for name, r in WAVLM_BASELINE.items():
    print(f'  WavLM {name:<20} {r["precision"]:>6.4f} {r["recall"]:>6.4f} {r["f1"]:>6.4f}')
print('-'*62)
for r in sorted(model_results, key=lambda x: -x['f1'])[:3]:
    print(f'  Text {r["model"]:<21} {r["precision"]:>6.4f} {r["recall"]:>6.4f} {r["f1"]:>6.4f}')
print('='*62)

best_wavlm_prec = max(r['precision'] for r in WAVLM_BASELINE.values())
best_wavlm_f1   = max(r['f1']        for r in WAVLM_BASELINE.values())
print()
print(f'Best WavLM precision: {best_wavlm_prec:.4f}   |   Winner text precision: {winner["precision"]:.4f}  (delta {winner["precision"]-best_wavlm_prec:+.4f})')
print(f'Best WavLM F1:        {best_wavlm_f1:.4f}   |   Winner text F1:        {winner["f1"]:.4f}  (delta {winner["f1"]-best_wavlm_f1:+.4f})')

## 12. Save Winner + Predictions for Fusion

In [ ]:
joblib.dump(winner['clf'],    SAVE_DIR / f"winner_{winner['model']}.pkl")
joblib.dump(winner['scaler'], SAVE_DIR / 'winner_scaler.pkl')

out = pd.DataFrame({'filename': filenames_te, 'label_int': y_te, 'text_proba': winner['proba']})
out.to_csv(SAVE_DIR / 'test_proba.csv', index=False)

abl_df.to_csv(SAVE_DIR / 'ablation.csv', index=False)
mdl_df.to_csv(SAVE_DIR / 'model_comparison.csv', index=False)
topN_df.to_csv(SAVE_DIR / 'topN_curve.csv', index=False)
imp.to_csv(SAVE_DIR / 'feature_importance.csv')

# audios4-CV per-fold diagnostics
if not abl_folds_df.empty:
    abl_folds_df.to_csv(SAVE_DIR / 'ablation_per_fold.csv', index=False)
if not mdl_folds_df.empty:
    mdl_folds_df.to_csv(SAVE_DIR / 'model_comparison_per_fold.csv', index=False)
if not topN_folds_df.empty:
    topN_folds_df.to_csv(SAVE_DIR / 'topN_per_fold.csv', index=False)

summary = {
    'train': TRAIN_FOLDERS, 'test': TEST_FOLDER,
    'n_features': len(feat_cols),
    'groups_used': list(GROUPS.keys()),
    'cv_protocol': {
        'target_batch': CV_TARGET_BATCH,
        'folds': CV_FOLDS,
        'deploy_pos_rate': DEPLOY_POS_RATE,
        'spw_deploy': round(SPW_DEPLOY, 4),
    },
    'winner': {k: (float(v) if isinstance(v,(int,float,np.floating)) else v)
               for k, v in winner.items() if k in (
                   'model','thr','precision','recall','f1','tp','fp','fn',
                   'cv_thr','cv_f1','cv_prec','cv_rec',
                   'test_f1_at_cv','test_prec_at_cv','test_rec_at_cv','gap_f1',
                   'fold_f1_mean','fold_f1_std','fold_thr_mean','fold_thr_std')},
    'best_ablation': abl_df.iloc[0].to_dict(),
    'best_topN':     topN_df.sort_values('f1', ascending=False).iloc[0].to_dict(),
}
with open(SAVE_DIR / 'summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)
print(f'Saved everything to {SAVE_DIR}/')
print(json.dumps(summary['winner'], indent=2))

## 13. Multi-threshold CV→test transfer (both configs)

Same analysis as `fusion_text_wavlm.ipynb` Section 6c. For each of 5 CV-chosen thresholds (best-F1, P80, P85, P90, P95) — freeze the CV threshold, apply to test, report test metric + gap.

Runs both:
- **A**: train=[audios2,audios4], CV=audios4, test=audios5
- **B**: train=[audios2,audios5], CV=audios5, test=audios4

Models covered:
- `ALL` — all features (8 groups, ~55 features)
- `only:<group>` — each feature group alone (disfluency, stylometric, pause, suspicious, formal_ai, prosodic, voice_q, perplexity)
- `topN:<K>` — top-K features by XGB importance (ranked on always-train batches only)

Columns per strategy: `thr`, `cv` (F1 or recall), `te` (same metric on test at frozen CV thr), `teP` (test precision at frozen CV thr — tells you if the precision floor held), `gap` = cv − te.

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

_STRATEGIES = ['F1','P80','P85','P90','P95']
_PREC_FLOOR = {'P80':0.80,'P85':0.85,'P90':0.90,'P95':0.95}

def _best_f1_thr_s13(p, y, grid=np.arange(0.20, 0.81, 0.01)):
    bt, bf = 0.5, -1.0
    for thr in grid:
        f = f1_score(y, (p >= thr).astype(int), zero_division=0)
        if f > bf: bf, bt = f, float(thr)
    return bt, bf

def _best_rec_at_prec_s13(p, y, target, min_tp=3):
    best = None
    for thr in np.arange(0.99, 0.10, -0.01):
        pred = (p >= thr).astype(int)
        cm = confusion_matrix(y, pred, labels=[0,1])
        if cm[1,1] < min_tp: continue
        pp = precision_score(y, pred, zero_division=0)
        rr = recall_score(y, pred, zero_division=0)
        if pp >= target and (best is None or rr > best[1]):
            best = (float(thr), float(rr), float(pp))
    return best if best is not None else (None, None, None)

def _pick_thr_s13(p, y, strategy):
    if strategy == 'F1':
        thr, f1 = _best_f1_thr_s13(p, y)
        return thr, f1
    thr, rec, _ = _best_rec_at_prec_s13(p, y, _PREC_FLOOR[strategy])
    return thr, rec

def _metrics_at_s13(p, y, thr):
    if thr is None: return dict(prec=None, rec=None, f1=None)
    pred = (p >= thr).astype(int)
    return dict(
        prec=float(precision_score(y, pred, zero_division=0)),
        rec =float(recall_score(y, pred, zero_division=0)),
        f1  =float(f1_score(y, pred, zero_division=0)),
    )

def _build_row_s13(name, p_cv, y_cv, p_te, y_te):
    row = {'model': name}
    for s in _STRATEGIES:
        thr, cv_val = _pick_thr_s13(p_cv, y_cv, s)
        te = _metrics_at_s13(p_te, y_te, thr)
        te_val = te['f1'] if s == 'F1' else te['rec']
        gap = (cv_val - te_val) if (cv_val is not None and te_val is not None) else None
        row[f'{s}_thr'] = round(thr, 2) if thr is not None else None
        row[f'{s}_cv']  = round(cv_val, 3) if cv_val is not None else None
        row[f'{s}_te']  = round(te_val, 3) if te_val is not None else None
        if s != 'F1':
            row[f'{s}_teP'] = round(te['prec'], 3) if te['prec'] is not None else None
        row[f'{s}_gap'] = round(gap, 3) if gap is not None else None
    return row

def _cols_s13():
    cols = ['model']
    for s in _STRATEGIES:
        cols += [f'{s}_thr', f'{s}_cv', f'{s}_te']
        if s != 'F1': cols += [f'{s}_teP']
        cols += [f'{s}_gap']
    return cols

def _xgb_factory_s13():
    return xgb.XGBClassifier(
        n_estimators=400, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
        scale_pos_weight=float(SPW_DEPLOY), eval_metric='logloss',
        random_state=RANDOM_SEED)

def run_config_s13(train_folders, cv_target, test_folder):
    tr_df = build_df(train_folders)
    te_df = build_df([test_folder])
    all_feat = [f for f in ALL_FEATURES if f in tr_df.columns]

    df_cv  = tr_df[tr_df['batch']==cv_target].reset_index(drop=True)
    df_al  = tr_df[tr_df['batch']!=cv_target].reset_index(drop=True)
    y_cv   = df_cv['label_int'].values
    y_al   = df_al['label_int'].values
    y_te   = te_df['label_int'].values

    def cv_and_test_for_cols(col_names):
        col_names = [c for c in col_names if c in all_feat]
        if not col_names: return None
        X_cv = df_cv[col_names].fillna(0).values
        X_al = df_al[col_names].fillna(0).values
        X_te = te_df[col_names].fillna(0).values

        oof = np.full(len(X_cv), np.nan)
        skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_SEED)
        for tr_i, va_i in skf.split(X_cv, y_cv):
            Xtr = np.vstack([X_al, X_cv[tr_i]]) if len(X_al) else X_cv[tr_i]
            ytr = np.concatenate([y_al, y_cv[tr_i]]) if len(X_al) else y_cv[tr_i]
            sc = StandardScaler().fit(Xtr)
            clf = _xgb_factory_s13()
            clf.fit(sc.transform(Xtr), ytr)
            oof[va_i] = clf.predict_proba(sc.transform(X_cv[va_i]))[:, 1]

        Xtr_full = np.vstack([X_al, X_cv]) if len(X_al) else X_cv
        ytr_full = np.concatenate([y_al, y_cv]) if len(X_al) else y_cv
        sc = StandardScaler().fit(Xtr_full)
        clf = _xgb_factory_s13()
        clf.fit(sc.transform(Xtr_full), ytr_full)
        p_te = clf.predict_proba(sc.transform(X_te))[:, 1]
        return dict(p_cv=oof, p_te=p_te)

    # Rank top-N on always-train batches only (no CV-target leakage)
    X_al_full = df_al[all_feat].fillna(0).values
    sc_r = StandardScaler().fit(X_al_full)
    ranker = _xgb_factory_s13()
    ranker.fit(sc_r.transform(X_al_full), y_al)
    imp = pd.Series(ranker.feature_importances_, index=all_feat).sort_values(ascending=False)

    rows = []
    r = cv_and_test_for_cols(all_feat)
    if r: rows.append(_build_row_s13('ALL', r['p_cv'], y_cv, r['p_te'], y_te))

    for gname, feats in GROUPS.items():
        r = cv_and_test_for_cols(feats)
        if r: rows.append(_build_row_s13(f'only:{gname}', r['p_cv'], y_cv, r['p_te'], y_te))

    for N in [5, 10, 15, 20, 30]:
        cols = imp.head(min(N, len(all_feat))).index.tolist()
        r = cv_and_test_for_cols(cols)
        if r: rows.append(_build_row_s13(f'topN:{N}', r['p_cv'], y_cv, r['p_te'], y_te))

    return pd.DataFrame(rows)[_cols_s13()], len(y_cv), len(y_te), int((y_cv==1).sum()), int((y_te==1).sum())

print('Running config A: train=[audios2,audios4] CV=audios4 test=audios5 ...')
df_A, nA_cv, nA_te, pA_cv, pA_te = run_config_s13(['audios2','audios4'], 'audios4', 'audios5')
print('Running config B: train=[audios2,audios5] CV=audios5 test=audios4 ...')
df_B, nB_cv, nB_te, pB_cv, pB_te = run_config_s13(['audios2','audios5'], 'audios5', 'audios4')

def _print_cfg(label, df, n_cv, n_te, p_cv, p_te):
    print('\n' + '='*110)
    print(label)
    print(f'  CV  : n={n_cv}  +{p_cv} / -{n_cv-p_cv}')
    print(f'  TEST: n={n_te}  +{p_te} / -{n_te-p_te}')
    print('='*110)
    with pd.option_context('display.max_columns', None, 'display.width', 220):
        print(df.to_string(index=False, na_rep='  --'))
    print('\n-- GAP TREND (cv - te) --')
    gap_cols = ['model'] + [f'{s}_gap' for s in _STRATEGIES]
    with pd.option_context('display.max_columns', None, 'display.width', 180):
        print(df[gap_cols].to_string(index=False, na_rep='  --'))

_print_cfg('CONFIG A — CV=audios4  test=audios5', df_A, nA_cv, nA_te, pA_cv, pA_te)
_print_cfg('CONFIG B — CV=audios5  test=audios4', df_B, nB_cv, nB_te, pB_cv, pB_te)

print('\nREADING: gap > 0 means CV was optimistic (test underperforms). '
      'teP < target (e.g. P95_teP < 0.95) means the precision floor did NOT hold on test.')
